In [2]:
!pip install -q groq openai


In [3]:

from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

SYSTEM = """
You are a helpful AI assistant for a grocery store.
You help customers with grocery-related questions.
Be friendly and creative when writing taglines.
"""

print("Grocery Store Assistant connected successfully!")


Grocery Store Assistant connected successfully!


In [4]:

def generate_tagline(temperature):
    for i in range(3):
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            temperature=temperature,
            messages=[
                {"role": "system", "content": SYSTEM},
                {"role": "user", "content": "Write a tagline for our Grocery Store."}
            ]
        )

        print(f"Attempt {i+1}:")
        print(response.choices[0].message.content)
        print("-" * 50)


print("TEMPERATURE = 0.2")
generate_tagline(0.2)

print("\nTEMPERATURE = 1.2")
generate_tagline(1.2)


TEMPERATURE = 0.2
Attempt 1:
**“Freshness, Flavor, and Friendly Smiles—All in One Aisle.”**
--------------------------------------------------
Attempt 2:
**“Fresh Finds, Friendly Faces – Your Neighborhood Grocery Hub!”**
--------------------------------------------------
Attempt 3:
**“Freshness, Flavor, and Friendly Faces – Every Aisle, Every Smile.”**
--------------------------------------------------

TEMPERATURE = 1.2
Attempt 1:
**“Fresh Finds, Friendly Faces, Every Day.”**
--------------------------------------------------
Attempt 2:
🌱 Fresh Finds, Friendly Smiles – Your Neighborhood Grocery Hub! 🌟
--------------------------------------------------
Attempt 3:
**"Freshness that Feels Like Home."**
--------------------------------------------------


In [5]:

# TASK 2: HALLUCINATION

# Remove the FACTS line intentionally for this experiment.
SYSTEM = """
You are the assistant for a Grocery Store.
Rules: Reply politely to customer questions.
Format: Reply in a short and friendly way.
"""

def ask(question):
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        temperature=0.2,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": question}
        ]
    )
    return response.choices[0].message.content


# Task 2 question
print(ask("Do you deliver to Kakinada?"))


Yes, we do deliver to Kakinada! Just let us know your address and we’ll sort out the details. 🚚✨


In [8]:

# TASK 3: NO MEMORY

# First call
print("First Call:")
print(ask("My name is kavitha"))

# Second call - separate API request
print("\nSecond Call:")
print(ask("What is my name?"))


First Call:
Nice to meet you, Kavitha! How can I help you today?

Second Call:
I’m sorry—I don’t have that information. Could you let me know your name?


In [11]:

# TASK 3: BOTH MESSAGES IN ONE REQUEST

messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "My name is kavitha"},
    {"role": "user", "content": "What is my name?"}
]

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    messages=messages
)

print("AI Response:")
print(response.choices[0].message.content)


AI Response:
Your name is Kavitha.


In [13]:

# TASK 3: MEMORY EXPERIMENT - CONTROLLED TEST

messages = [
    {"role": "system", "content": "You are a helpful assistant. Answer using only the messages in this request."},
    {"role": "user", "content": "My name is kavitha."},
    {"role": "user", "content": "What is my name?"}
]

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    messages=messages
)

print("AI Response:")
print(response.choices[0].message.content)


AI Response:
Your name is Kavitha.


In [14]:

# TASK 4: TOKEN COMPARISON

# English question
english_response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    messages=[
        {"role": "user", "content": "What are the benefits of eating fresh fruits?"}
    ]
)

# Telugu question
telugu_response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    messages=[
        {"role": "user", "content": "తాజా పండ్లు తినడం వల్ల కలిగే ప్రయోజనాలు ఏమిటి?"}
    ]
)

# Display token usage
print("English Question:")
print("Prompt Tokens:", english_response.usage.prompt_tokens)

print("\nTelugu Question:")
print("Prompt Tokens:", telugu_response.usage.prompt_tokens)


English Question:
Prompt Tokens: 80

Telugu Question:
Prompt Tokens: 92


In [15]:

# TASK 5: MY GROCERY STORE ASSISTANT

SYSTEM = """
You are a friendly AI assistant for a Grocery Store.

FACTS:
1. The store sells groceries and daily household essentials.
2. The store provides food items and packaged products.
3. Customers can ask about available grocery products.
4. Product prices depend on the selected items.
5. Customers can contact the store to confirm product availability.

RULES:
1. Never invent product prices or stock availability.
2. Never promise delivery unless it is confirmed.
3. If information is unknown, politely ask the customer to contact the store.

Keep every response short, friendly and helpful.
"""

def ask(question):
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        temperature=0.2,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": question}
        ]
    )
    return response.choices[0].message.content


In [16]:

# Ask 3 real customer questions

print("Customer 1:")
print(ask("Do you have rice and cooking oil?"))

print("\nCustomer 2:")
print(ask("What is the price of 1 kg of rice?"))

print("\nCustomer 3:")
print(ask("Do you provide home delivery?"))


Customer 1:
I’m not sure about the current stock. Please give us a call or visit the store to confirm availability. Happy to help!

Customer 2:
I’m not sure about the current price of 1 kg of rice. Please give us a call or visit the store to confirm the exact cost. Thanks!

Customer 3:
I’m not sure about delivery options right now. Please give the store a call or check their website for the most up‑to‑date information. Happy to help with anything else!
